# Lumbar Spine L4 Extraction on AP Views — BUU-LSPINE + YOLOv8

Instead of using SpineFM (which is lateral-view specific), this pipeline is tailored specifically for the Anteroposterior (AP) lumbar spine domain.

Pipeline Flow:
1. Label Conversion: Convert BUU-LSPINE AP labels (L1–L5 corner coordinates) into the standard YOLO format.
2. Model Training: Train a YOLOv8 object detection model using 5 distinct classes (L1 to L5).
3. Inference & Extraction: Apply the trained model to your custom AP DICOM images $\rightarrow$ Identify the 4th vertebra from the top (L4) $\rightarrow$ Extract (Bounding Box + Crop + Overlay).

Dataset Links:
 - Official Page: https://services.informatics.buu.ac.th/spine/#sq-tab1
 - GitHub Repository: https://github.com/North-Github/BUU-LSPINE (AP+LA, 7,200 images, L1–L5 labels)
 - Dataset download: https://drive.google.com/file/d/1OqM9J8S2Cy23D-4nIRJdZdzD9-mmo2Og/view
 - Theses: https://www.mdpi.com/2076-3417/13/15/8646


## 0. Install

In [1]:
# !pip install ultralytics pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg opencv-python-headless
import os, glob, re, random, shutil
import numpy as np
from PIL import Image
import cv2
print("ready")

ready


## 1. Path Configuration
Download and extract the BUU-LSPINE dataset, then specify the directory paths for the AP images and AP labels.

In [2]:
# ---- BUU-LSPINE (Training Data) ----
BUU_AP_IMAGES = "./BUU-LSPINE-400/AP/images"   # AP X-ray image folder (.jpg/.png)
BUU_AP_LABELS = "./BUU-LSPINE-400/AP/labels"   # AP label (.txt) folder (same stem as image)

# ---- YOLO training settings ----
YOLO_ROOT = "./yolo_ap_lspine"
VAL_RATIO = 0.15
SEED = 42

#Class: From top to bottom L1..L5 (index 0..4). L4 = index 3.
CLASS_NAMES = ["L1", "L2", "L3", "L4", "L5"]
L4_CLASS_ID = CLASS_NAMES.index("L4")

# Number of 'edge lines' forming each vertebra (superior/inferior endplates = 2)
LINES_PER_VERTEBRA = 2
N_VERTEBRAE = 5
random.seed(SEED)
print("L4 class id =", L4_CLASS_ID)

L4 class id = 3


## 2. Label Format Check (Run First!)

Print an actual BUU-LSPINE label file as is. Visually verify if it matches the parser assumptions below.

- Assumption: One file contains **10 lines (edge ​​lines)**, with 4 numbers per line = `(x_left, y_left, x_right, y_right)`.

- Two consecutive lines = one vertebral body (upper/lower endplate). A total of 5 vertebral bodies.

In [3]:
lbls = sorted(glob.glob(os.path.join(BUU_AP_LABELS, "*")))
print("File for Labels:", len(lbls))
if lbls:
    print(lbls[0], "\n----- Original Content -----")
    with open(lbls[0]) as f:
        print(f.read())

File for Labels: 400
./BUU-LSPINE-400/AP/labels\0001-F-037Y0.csv 
----- Original Content -----
876.2222,167.0618,1111.472,168.7665,0
865.9939,313.6671,1119.996,313.6671,0
870.8508,344.4474,1116.587,351.1707,0
847.6437,498.0999,1119.997,511.2078,0
842.3262,553.4754,1108.814,557.2025,0
825.0808,705.7509,1118.291,724.5027,0
826.1249,774.749,1111.775,787.0792,0
801.4645,937.0969,1113.83,957.6473,0
797.71,983.067,1115.284,980.1932,0
766.5613,1163.979,1129.421,1157.696,0



## 3. Label Parser + YOLO Transformation

Converts the 4 corner points of each vertebra into bounding boxes (min/max).

**Align the 5 vertebrae y-axis and assign them as L1..L5 from top to bottom.**

(Safe regardless of whether the files are arranged from top to bottom or bottom to top).

In [4]:
def _read_floats_per_line(path):
    rows = []
    with open(path) as f:
        for line in f:
            nums = re.findall(r"-?\d+\.?\d*", line)
            if len(nums) >= 4:
                rows.append([float(v) for v in nums[:4]])  # xL,yL,xR,yR
    return rows  # 기대: 10줄

def parse_annotation(path):
    """Label file -> List of bboxes (pixels) in L1..L5 order. [] on failure"""
    rows = _read_floats_per_line(path)
    need = LINES_PER_VERTEBRA * N_VERTEBRAE
    if len(rows) < need:
        return []
    verts = []
    for i in range(N_VERTEBRAE):
        pts = []
        for j in range(LINES_PER_VERTEBRA):
            xL, yL, xR, yR = rows[i*LINES_PER_VERTEBRA + j]
            pts += [(xL, yL), (xR, yR)]
        xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        verts.append((y1, (x1, y1, x2, y2)))     # For sorting by y1
    verts.sort(key=lambda v: v[0])                #Up -> Down
    return [b for _, b in verts]                  # bbox list in order L1..L5

def to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2 / W; cy = (y1 + y2) / 2 / H
    w = (x2 - x1) / W;      h = (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"

def find_image(stem):
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
        p = os.path.join(BUU_AP_IMAGES, stem + ext)
        if os.path.exists(p):
            return p
    return None

# ---- Run conversion ----
for sub in ["images/train","images/val","labels/train","labels/val"]:
    os.makedirs(os.path.join(YOLO_ROOT, sub), exist_ok=True)

label_files = sorted(glob.glob(os.path.join(BUU_AP_LABELS, "*")))
random.shuffle(label_files)
n_val = int(len(label_files) * VAL_RATIO)
ok = skip = 0
for k, lp in enumerate(label_files):
    stem = os.path.splitext(os.path.basename(lp))[0]
    ip = find_image(stem)
    if ip is None:
        skip += 1; continue
    boxes = parse_annotation(lp)
    if len(boxes) != N_VERTEBRAE:
        skip += 1; continue
    im = cv2.imread(ip); H, W = im.shape[:2]
    split = "val" if k < n_val else "train"
    shutil.copy(ip, os.path.join(YOLO_ROOT, "images", split, os.path.basename(ip)))
    with open(os.path.join(YOLO_ROOT, "labels", split, stem + ".txt"), "w") as f:
        for cid, box in enumerate(boxes):           # cid 0..4 = L1..L5
            f.write(to_yolo_line(cid, box, W, H) + "\n")
    ok += 1
print(f"Conversion complete: {ok}pages, Skip {skip}pages (val {n_val})")

Conversion complete: 400pages, Skip 0pages (val 60)


In [5]:
import os
import glob
import cv2
import math
import matplotlib.pyplot as plt

# 1. Path setting
BUU_AP_IMAGES = "./BUU-LSPINE/AP/images"
BUU_AP_LABELS = "./BUU-LSPINE/AP/labels"

# 2. Get All labels
lbls = sorted(glob.glob(os.path.join(BUU_AP_LABELS, "*")))

if not lbls:
    print("The Label file Cannot be found. Please Check the path.")
else:
    # ----------------------------------------------------
    # Setting the Range [1:100]
    START_IDX = 1
    END_IDX = 100
    
    selected_lbls = lbls[START_IDX:END_IDX]
    # ----------------------------------------------------
    
    total_plots = len(selected_lbls)
    print(f"Total {total_plots} images (Index from {START_IDX} to {END_IDX-1})")

    if total_plots == 0:
        print("There is no label file.")
    else:
        # For grid coordinates
        cols = min(4, total_plots)
        rows = math.ceil(total_plots / cols)
        
        # Automatically adjust figure sie
        plt.figure(figsize=(cols * 4, rows * 5))
        
        plot_idx = 1
        for label_path in selected_lbls:
            stem = os.path.splitext(os.path.basename(label_path))[0]
            
            # 3. Finding the corresponding image file
            sample_image_path = None
            for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
                p = os.path.join(BUU_AP_IMAGES, stem + ext)
                if os.path.exists(p):
                    sample_image_path = p
                    break
                    
            if sample_image_path is None:
                # if there is no image, create an empty subplot space and display a skip message
                plt.subplot(rows, cols, plot_idx)
                plt.text(0.5, 0.5, f"No Image\n{stem}", ha='center', va='center', color='red')
                plt.axis('off')
                plot_idx += 1
                continue
                
            # 4. read the value of boxes 
            boxes = parse_annotation(label_path)
            
            # 5. Image Reading and RGB Conversion
            img = cv2.imread(sample_image_path)
            if img is None:
                plot_idx += 1
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # 6. Draw the bounding box of each vertebra as a red rectangle
            if boxes:
                for box in boxes:
                    x1, y1, x2, y2 = [int(v) for v in box]
                    # Increased line thickness to 4 to make it more visible when visualizing multiple images.
                    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 0, 0), 4)
            
            # 7. Place images in subplot positions within the grid
            plt.subplot(rows, cols, plot_idx)
            plt.imshow(img_rgb)
            plt.title(f"[{START_IDX + plot_idx - 1}] {stem}", fontsize=9)
            plt.axis('off')
            
            plot_idx += 1
            
        plt.tight_layout()
        plt.show()

The Label file Cannot be found. Please Check the path.


## 4. Create data.yaml

In [6]:
yaml_path = os.path.join(YOLO_ROOT, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {os.path.abspath(YOLO_ROOT)}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write(f"nc: {len(CLASS_NAMES)}\n")
    f.write(f"names: {CLASS_NAMES}\n")
print(open(yaml_path).read())

path: c:\Users\csm02\Desktop\edward\bmd\1 src\yolo_ap_lspine
train: images/train
val: images/val
nc: 5
names: ['L1', 'L2', 'L3', 'L4', 'L5']



## 5. YOLOv8 Training

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")          # more accurately : yolov8s.pt / yolov8m.pt
results = model.train(data=yaml_path, 
            epochs=100, 
            imgsz=640, 
            batch=-1,
            project="ap_lspine", 
            name="l1_l5", 
            patience=20, 
            workers=4)
BEST = "ap_lspine/l1_l5/weights/best.pt"
print("best weights ->", BEST)

New https://pypi.org/project/ultralytics/8.4.68 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.12.13 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_ap_lspine\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mo

## 6. DICOM Inference → L4 Extraction
Convert DICOM to a brightness similar to BUU-LSPINE (typically a standard radiographic image where the bone is bright).

Adjust the brightness using `INVERT` (so that the tone matches the training images).

In [ ]:
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

DICOM_DIR  = "./dataset-dcm/train/gather/"
OUT_DIR    = "./l4_output_ap"
BEST = os.path.join(results.save_dir, "weights", "best.pt")
INVERT     = True       # If BUU-LSPINE is a bone-bright tone, invert MONOCHROME1.
CONF       = 0.25

os.makedirs(OUT_DIR, exist_ok=True)
det = YOLO(BEST)

def dicom_to_rgb(path, invert=INVERT):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    arr = arr*float(getattr(ds,"RescaleSlope",1.0)) + float(getattr(ds,"RescaleIntercept",0.0))
    try: arr = apply_voi_lut(arr.astype(ds.pixel_array.dtype), ds).astype(np.float32)
    except Exception: pass
    if invert and str(getattr(ds,"PhotometricInterpretation","")).upper()=="MONOCHROME1":
        arr = arr.max()-arr
    lo,hi = np.percentile(arr,0.5), np.percentile(arr,99.5)
    if hi<=lo: lo,hi = arr.min(), arr.max()+1e-6
    arr = np.clip((arr-lo)/(hi-lo),0,1)
    return cv2.cvtColor((arr*255).astype(np.uint8), cv2.COLOR_GRAY2RGB)

def pick_l4(boxes, classes, confs):
    """1순위: L4 클래스 박스. 없으면 위->아래 정렬 4번째."""
    if len(boxes)==0: return None, "검출 없음"
    idx = [i for i,c in enumerate(classes) if c==L4_CLASS_ID]
    if idx:
        b = max(idx, key=lambda i: confs[i])
        return boxes[b], "L4 클래스 직접"
    order = sorted(range(len(boxes)), key=lambda i: (boxes[i][1]+boxes[i][3])/2)  # y center
    if len(order)>=4:
        return boxes[order[3]], "위->아래 4번째(폴백)"
    return None, f"척추체 {len(boxes)}개뿐"

ok=fail=0
for path in sorted(glob.glob(os.path.join(DICOM_DIR,"*.dcm"))):
    stem = os.path.splitext(os.path.basename(path))[0]
    try:
        rgb = dicom_to_rgb(path)
        r = det.predict(rgb, conf=CONF, verbose=False)[0]
        boxes = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        box, status = pick_l4(boxes, classes, confs)
        if box is None:
            print(f"[skip] {stem[:14]}: {status}"); fail+=1; continue
        x1,y1,x2,y2 = [int(v) for v in box]
        crop = rgb[max(0,y1):y2, max(0,x1):x2]
        cv2.imwrite(os.path.join(OUT_DIR, f"{stem}_L4_crop.png"), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        np.save(os.path.join(OUT_DIR, f"{stem}_L4_bbox.npy"), np.array([x1,y1,x2,y2]))
        vis = rgb.copy(); cv2.rectangle(vis,(x1,y1),(x2,y2),(0,255,0),2)
        cv2.putText(vis,"L4",(x1,max(0,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
        cv2.imwrite(os.path.join(OUT_DIR, f"{stem}_L4_overlay.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        print(f"[ ok ] {stem[:14]}: {status} | L4 bbox=({x1},{y1},{x2},{y2})"); ok+=1
    except Exception as e:
        print(f"[fail] {stem[:14]}: {e}"); fail+=1
print(f"\n완료. L4 {ok}건 / 실패 {fail}건 -> {OUT_DIR}")

[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(94,102,130,121)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(101,43,146,64)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(112,110,150,130)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(106,19,137,31)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(101,108,139,127)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(109,136,148,158)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(109,85,151,105)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(114,86,156,103)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(111,40,143,55)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(98,81,137,98)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(112,122,156,146)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(90,90,142,111)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(101,84,143,103)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(122,103,171,120)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(98,0,154,12)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(108,90,153,110)
[ ok ] 1.2.826.0.1.36: L4 클래스 직접 | L4 bbox=(

## Reference
- If a **pixel mask** is needed, train with **YOLOv8-seg** (`yolov8n-seg.pt`) instead of detect.
  However, since BUU-LSPINE is a corner point label, it becomes a rectangular polygon mask, and a separate mask label is required for precise segmentation.
- Make sure the inference tone (`INVERT`) matches the brightness of the training image (use check cell 2)
- If detection is weak: Check for scaling up to yolov8s/m, increasing epochs, and data augmentation (mosaic, etc.).

In [ ]:
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))